[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/junlinghu/book-agents/blob/main/tutorial/8-verification.ipynb)

## Colab setup

On Google Colab, use the setup in `tutorial/common/colab.py`. A fresh Colab runtime does not have this repository, so clone the repo into the session first. Then, before the lesson cells, run `setup_colab` from that module (import it from `tutorial.common.colab` and call it, or open the file and run `setup_colab` from there). The helper installs the tutorial packages, checks the repo out under `/content/book-agents` when it is still missing, and copies a Colab secret named `OPENAI_API_KEY`. Local Jupyter and VS Code can skip it when you already have the repo.

An API key is required. Set `OPENAI_API_KEY` in Colab secrets (the key icon, secret name `OPENAI_API_KEY`) before you run `setup_colab`. Locally, put the same name in the repository-root `.env`. A missing key stops the notebook. The key is not printed.

Edits you make in Colab stay in that session. They do not push to GitHub.

# 8. Verification

The loop can sound sure and still be wrong. A checker is a second function with the rules written in code.

`verify_proposal` loads the shelf row itself. Quantity must be the gap. Shipping dairy fails. An empty citation fails. The notebook runs a bad proposal and a good one before the model is involved, then lets the agent call the checker. The shelf and the loop are imported above.

Shared helpers this lesson needs are imported, not copied from earlier notebooks. You can run this file by itself.

## Setup

This notebook calls the OpenAI Chat Completions API. `OPENAI_API_KEY` is required. Locally the key is loaded from the repository-root `.env` by `tutorial/common/client.py`. On Colab, `setup_colab` in `tutorial/common/colab.py` reads a secret of the same name. The value is not printed. A missing key stops the run.

Companion notes: `8-verification.md`.

In [ ]:
import sys
from pathlib import Path


def find_root():
    """Repo root, whether the kernel started here or in tutorial."""
    here = Path.cwd().resolve()
    for candidate in [here, *here.parents]:
        marker = candidate / "tutorial" / "common" / "client.py"
        if marker.is_file():
            return candidate
    raise RuntimeError(
        "Open this notebook inside the book-agents repository. "
        "The setup cell looks for tutorial/common/client.py."
    )


ROOT = find_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from tutorial.common.client import describe_runtime
from tutorial.runtime import chat, require_key

LESSON = '8-verification'
LESSON_NUMBER = 8
require_key()
print("tutorial:", LESSON)
print(describe_runtime())


## Shared pieces

These imports are the earlier pieces this lesson uses. They come from `tutorial.common` and `tutorial.runtime`. You do not need to run the previous notebook first.


In [ ]:
import json

from tutorial.common.harness import check, system_text
from tutorial.common.loop import run_agent
from tutorial.common.shelf import reset_db


## Tutorial 8 — a separate checker

The proposer and the checker are different functions. A bad quantity fails here.

In [ ]:
import json

from tutorial.common.harness import register
from tutorial.common.shelf import gap_for, row_for


def verify_proposal(args):
    """Separate checker. It does not trust qty, ship, or a citation written by the proposer."""
    sku = str(args.get("sku", "")).strip()
    row = row_for(sku)
    findings = []
    expected = None
    if row is None:
        findings.append("unknown_sku")
    else:
        expected = gap_for(row)
        try:
            qty = int(args.get("qty"))
        except (TypeError, ValueError):
            qty = None
            findings.append("bad_qty")
        if qty is not None and qty != expected:
            findings.append("qty_must_be_" + str(expected))
        ship = bool(args.get("ship", False))
        blocked = row["category"] in {"dairy", "bakery"} or "milk" in row["name"].lower()
        if ship and blocked:
            findings.append("not_shippable")
    citation = str(args.get("citation", "")).strip()
    if not citation:
        findings.append("missing_citation")
    status = "rejected" if findings else "accepted"
    return json.dumps({
        "status": status,
        "sku": sku,
        "expected_qty": expected,
        "findings": findings,
    })


register(
    "verify_proposal",
    "Check a restock proposal. qty must be the shelf gap (par minus on_hand). "
    "ship must be false for dairy and bakery. citation must be non-empty. "
    "This checker is not the same step as the proposal.",
    {
        "sku": {"type": "string"},
        "qty": {"type": "integer"},
        "ship": {"type": "boolean"},
        "citation": {"type": "string"},
    },
    ["sku", "qty"],
    verify_proposal,
)


## Run

`system_text()` mentions only the tools imported above. Run this cell after the ones above. Each model turn calls the Chat Completions API. A check prints a warning when the wording differs from the expected trace, and the notebook continues.

In [ ]:
reset_db()
print("=== checker on two proposals, no model required ===")
bad = verify_proposal({"sku": "OM-32", "qty": 1000, "ship": True, "citation": ""})
good_row = row_for("OM-32")
good = verify_proposal({
    "sku": "OM-32",
    "qty": good_row["gap"],
    "ship": False,
    "citation": "inventory",
})
print(bad)
print(good)
check(json.loads(bad)["status"] == "rejected", "an invented quantity and a ship flag fail")
check("not_shippable" in bad and "missing_citation" in bad, "ship and citation rules both fire")
check(json.loads(good)["status"] == "accepted", "the shelf gap passes the checker")

print("=== the agent may propose; the checker decides ===")
result = run_agent(
    "Propose an oat milk restock and have the checker score it. Do not ship it.",
    system_text(),
)
print("ANSWER:", result["text"])
check(result["stopped"] == "final", "verify run finished")
check("query_inventory" in "\n".join(result["tool_log"]), "the proposal looked at the shelf")
check("verify_proposal" in "\n".join(result["tool_log"]), "a separate checker ran")
check("accepted" in result["text"].lower(), "the reported proposal is the one the checker accepted")
